# Global Player Mobility in Football

## Analysis (from prepared edge lists)

In [15]:
from pathlib import Path
import pandas as pd
import igraph as ig
import leidenalg
import numpy as np

PROJECT_ROOT = Path.cwd()

# Default analysis input
EDGE_ALL_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_all_strict.csv'
EDGE_SEASON_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_season_strict.csv'

# Alternative (with unknown competitions kept)
# EDGE_ALL_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_all_with_unknown.csv'
# EDGE_SEASON_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_season_with_unknown.csv'

edge_all = pd.read_csv(EDGE_ALL_PATH)
edge_season = pd.read_csv(EDGE_SEASON_PATH)

pd.DataFrame({
    'table': ['edge_all', 'edge_season'],
    'rows': [len(edge_all), len(edge_season)],
    'cols': [edge_all.shape[1], edge_season.shape[1]],
})

,table,rows,cols
0,edge_all,31382,13
1,edge_season,115229,14


## 05b  Graph Construction
Build directed graphs from the edge lists

In [2]:
src_labels = edge_all[['source_league_id', 'source_league']].dropna().drop_duplicates()
dst_labels = edge_all[['dest_league_id', 'dest_league']].dropna().drop_duplicates()
src_labels.columns = ['league_id', 'league_name']
dst_labels.columns = ['league_id', 'league_name']
league_labels = (
    pd.concat([src_labels, dst_labels], ignore_index=True)
    .drop_duplicates(subset=['league_id'])
    .set_index('league_id')['league_name']
    .to_dict()
)

agg_tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in edge_all.itertuples(index=False)]
G_all = ig.Graph.TupleList(agg_tuples, directed=True, weights=True)
G_all.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in G_all.es['weight']]
G_all.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in G_all.vs]

graphs_by_season = {}
for season, df in edge_season.groupby('season'):
    tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in df.itertuples(index=False)]
    g = ig.Graph.TupleList(tuples, directed=True, weights=True)
    g.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in g.es['weight']]
    g.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in g.vs]
    graphs_by_season[season] = g

G_all.vcount(), G_all.ecount()

(650, 31382)

## 6 Network Metrics 

## 6a Basic Metrics: weighted in/out strength, degree, and betweenness for the aggregated graph

In [3]:
metrics = pd.DataFrame({
    'league': G_all.vs['name'],
    'league_name': G_all.vs['league_name'],
    'in_strength': G_all.strength(mode='in', weights='weight'),
    'out_strength': G_all.strength(mode='out', weights='weight'),
    'in_degree': G_all.degree(mode='in'),
    'out_degree': G_all.degree(mode='out'),
    # Betweenness uses distance, so we invert transfer weights in graph construction
    'betweenness': G_all.betweenness(directed=True, weights='distance'),
})

metrics.sort_values('in_strength', ascending=False).head(15)

,league,league_name,in_strength,out_strength,in_degree,out_degree,betweenness
20,WITHOUT_CLUB,WITHOUT_CLUB,45886.0,33456.0,583,576,383994.166667
91,RETIREMENT,RETIREMENT,16905.0,58.0,476,32,5607.000000
135,IT1,Serie A,16002.0,16600.0,193,172,652.000000
84,IT2,Serie B,12539.0,13105.0,170,153,1841.000000
196,GB2,Championship,11429.0,12195.0,183,174,4164.000000
174,BRA1,Campeonato Brasileiro Série A,10988.0,12437.0,164,181,9266.000000
31,GB3,League One,9479.0,9939.0,129,139,8646.000000
94,TR1,Süper Lig,8756.0,9255.0,179,162,581.000000
195,GB1,Premier League,8698.0,8987.0,140,121,1223.000000
175,BRA2,Campeonato Brasileiro Série B,8594.0,8823.0,166,173,29238.000000


## 06b Communities
We will use Leidenalg here for community detection

In [4]:
parts = leidenalg.find_partition(
    G_all,
    leidenalg.RBConfigurationVertexPartition,
    weights='weight',
    seed=42,
)

communities = {G_all.vs[idx]['name']: int(parts.membership[idx]) for idx in range(G_all.vcount())}
metrics['community'] = metrics['league'].map(communities)
'leiden'

'leiden'

## 07 Hypothesis Tests & Robustness

## 7a - H1 – Concentration: Incoming transfer flows are highly concentrated, with a small set of leagues accounting for a large share of all incoming transfers.

In [ ]:
inflow = (edge_all.groupby('dest_league_id', dropna=False)
    .agg(total_in_transfers=('n_transfers','sum'))
    .reset_index()
    .sort_values('total_in_transfers', ascending=False))

name_map = metrics.set_index('league')['league_name'].to_dict()
inflow['league_name'] = inflow['dest_league_id'].map(name_map).fillna(inflow['dest_league_id'])

inflow.head(20)

,dest_league_id,total_in_transfers,league_name
626,WITHOUT_CLUB,45886,WITHOUT_CLUB
496,RETIREMENT,16905,RETIREMENT
292,IT1,16002,Serie A
293,IT2,12539,Serie B
246,GB2,11429,Championship
84,BRA1,10988,Campeonato Brasileiro Série A
248,GB3,9479,League One
569,TR1,8756,Süper Lig
244,GB1,8698,Premier League
85,BRA2,8594,Campeonato Brasileiro Série B


In [8]:
total_in = inflow['total_in_transfers'].sum()
top5_share = inflow.head(5)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
top10_share = inflow.head(10)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
shares = inflow['total_in_transfers'] / total_in if total_in > 0 else inflow['total_in_transfers']
hhi = (shares ** 2).sum()

pd.DataFrame({
    'total_in_transfers': [total_in],
    'top5_share': [top5_share],
    'top10_share': [top10_share],
    'hhi': [hhi],
}).round(4)


,total_in_transfers,top5_share,top10_share,hhi
0,586520,0.1752,0.2545,0.014


## 7b - H2 – Bridges: Some mid-level leagues exhibit disproportionately high betweenness centrality and act as bridges between regions and/or tier systems.

In [10]:
tier_map_src = edge_all[['source_league_id','source_tier']].dropna().drop_duplicates().rename(columns={'source_league_id':'league','source_tier':'tier'})
tier_map_dst = edge_all[['dest_league_id','dest_tier']].dropna().drop_duplicates().rename(columns={'dest_league_id':'league','dest_tier':'tier'})
tier_map = pd.concat([tier_map_src, tier_map_dst], ignore_index=True).drop_duplicates(subset=['league'])
tier_map_dict = tier_map.set_index('league')['tier'].to_dict()

metrics['tier'] = metrics['league'].map(tier_map_dict).fillna('Unknown')
metrics['tier_num'] = pd.to_numeric(metrics['tier'], errors='coerce')
metrics['is_mid_tier'] = metrics['tier_num'].between(2, 4, inclusive='both').fillna(False)

metrics[['league','league_name','tier','is_mid_tier','betweenness']].head()

,league,league_name,tier,is_mid_tier,betweenness
0,17LA,U17 DFB-Nachwuchsliga - Hauptrunde Liga A,Youth,False,0.0
1,19LA,U19 DFB-Nachwuchsliga - Hauptrunde Liga A,Youth,False,623.0
2,17LB,U17 DFB-Nachwuchsliga - Hauptrunde Liga B,Youth,False,0.0
3,19LB,U19 DFB-Nachwuchsliga - Hauptrunde Liga B,Youth,False,624.0
4,181F,U18 Divisie 1 Herbst,Youth,False,1278.0


In [11]:
bridge_rank = (metrics
    .sort_values('betweenness', ascending=False)
    [['league','league_name','tier','is_mid_tier','betweenness','in_strength','out_strength']])
bridge_rank.head(25)

,league,league_name,tier,is_mid_tier,betweenness,in_strength,out_strength
20,WITHOUT_CLUB,WITHOUT_CLUB,Unknown,False,383994.166667,45886.0,33456.0
175,BRA2,Campeonato Brasileiro Série B,2,True,29238.000000,8594.0,8823.0
155,A2,2. Liga,2,True,27094.000000,2708.0,2802.0
336,ARG2,Primera Nacional,2,True,21373.500000,5424.0,5734.0
215,L3,3. Liga,3,True,17090.666667,4786.0,4952.0
15,NL2,Keuken Kampioen Divisie,2,True,16786.000000,4165.0,4640.0
53,RLW3,Regionalliga West,4,True,14977.666667,2326.0,2579.0
77,C2,Challenge League,2,True,14072.000000,3013.0,3122.0
121,RU2,1.Division,2,True,13751.000000,5767.0,5925.0
368,MEX2,Liga de Expansión MX Clausura,2,True,13510.000000,4155.0,4211.0


In [12]:
k = 25
topk = bridge_rank.head(k)
mid_share_topk = topk['is_mid_tier'].mean() if len(topk) else 0
mid_share_all = metrics['is_mid_tier'].mean() if len(metrics) else 0

pd.DataFrame({
    'k': [k],
    'mid_share_topk_betweenness': [mid_share_topk],
    'mid_share_all_leagues': [mid_share_all],
    'lift_topk_vs_all': [mid_share_topk / mid_share_all if mid_share_all > 0 else None],
}).round(4)

,k,mid_share_topk_betweenness,mid_share_all_leagues,lift_topk_vs_all
0,25,0.8,0.3815,2.0968


## 07c - H3 – Communities: The network exhibits non-random community structure, with detected communities showing higher modularity and/or greater stability than in a randomized baseline network.

In [13]:
observed_modularity = parts.quality()
observed_modularity

351360.49653208756

In [16]:
n_runs = 30
niter_factor = 10  # rewiring intensity relative to edge count
null_modularity = []

for run in range(n_runs):
    g_null = G_all.copy()
    n_rewire = int(max(1, g_null.ecount() * niter_factor))

 
    try:
        g_null.rewire(n=n_rewire, allowed_edge_types='simple')
    except TypeError:
        # Backward-compatible API
        g_null.rewire(n=n_rewire, mode='simple')

    # For null graphs, use unweighted Leiden to avoid version/weight edge-cases
    null_parts = leidenalg.find_partition(
        g_null,
        leidenalg.RBConfigurationVertexPartition,
        seed=42 + run,
    )
    null_modularity.append(null_parts.quality())

null_modularity = np.array(null_modularity)
pd.Series(null_modularity).describe()

count      30.000000
mean     2508.304360
std        35.896055
min      2440.803709
25%      2486.609585
50%      2506.533873
75%      2529.726675
max      2576.440157
dtype: float64

In [ ]:
null_mean = float(null_modularity.mean())
null_std = float(null_modularity.std(ddof=1)) if len(null_modularity) > 1 else np.nan
z_score = (observed_modularity - null_mean) / null_std if null_std and not np.isnan(null_std) else np.nan
p_one_sided = float((null_modularity >= observed_modularity).mean())

pd.DataFrame({
    'observed_modularity': [observed_modularity],
    'null_mean_modularity': [null_mean],
    'null_std_modularity': [null_std],
    'z_score_vs_null': [z_score],
    'p_value_one_sided': [p_one_sided],
}).round(4)


,observed_modularity,null_mean_modularity,null_std_modularity,z_score_vs_null,p_value_one_sided
0,351360.4965,2508.3044,35.8961,9718.3991,0.0


: 

## 08 Visuals 

In [5]:
# TODO: heatmaps and community/network visuals
pass

## 09 Results Summary (stub)

In [6]:
# TODO: final tables + short interpretation
pass